# Engenharia de Freatures: Goodreads Dataset
Esta etapa transforma o dataset limpo em uma **estrutura analiticamente granular**, capaz de responder perguntas por gênero literário individual. A principal intervenção é a normalização da coluna `genres`, que no formato original agrupa múltiplos gêneros por registro — impedindo qualquer análise de desempenho por categoria.

- **Escopo:** Desnormalização da coluna `genres` para o modelo um-gênero-por-linha
- **Produto desta etapa:** `goodreads_books_exploded.csv` — estrutura otimizada para análises de popularidade e avaliação por gênero

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..', '..')))


# Importação de Bibliotecas e Módulos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
import src.io.data_loader as dl
import src.io.data_save as ds
import src.view.tables as tb
import src.utils.features as feat

---

## Carregando Dados

In [2]:
caminho = '../../data/interim/books/goodreads_books_cleaned.parquet'
df_books = dl.load_data(caminho, tipo_arquivo='parquet')

Dados Parquet carregados! Formato: (10897, 12)


---
## Harmonização e Padronização de Notas

In [3]:
# Harmonização do esquema de dados para o domínio de livros
df_books = feat.harmonizar_esquema_dados(df_books, 'livro')

In [4]:
# Criação de features globais para o domínio de livros
df_books = feat.criar_features_globais(df_books, 5)

In [5]:
display(tb.estilizar_tabela(df_books, qtd_linhas=10))

,book_id,title,author,average_rating,isbn,original_language,num_pages,total_votes,text_reviews_count,release_date,producer_company,genres,global_score,release_year,decade,age_years,votes_per_year,popularity_tier
0,1,Harry Potter and the Half-Blood Prince (Harry Potter #6),J.K. Rowling/Mary GrandPré,4.57,0439785960,eng,652 págs,"2,095,690",27591,2006-09-16 00:00:00,Scholastic Inc.,"['Fantasy' 'Young Adult' 'Fiction' 'Fantasy,Magic' 'Childrens' 'Adventure' 'Audiobook' 'Childrens,Middle Grade' 'Classics' 'Science Fiction Fantasy']",91.40,2006,2000,20 anos,"104,784",Mainstream Hit
1,2,Harry Potter and the Order of the Phoenix (Harry Potter #5),J.K. Rowling/Mary GrandPré,4.49,0439358078,eng,870 págs,"2,153,167",29221,2004-09-01 00:00:00,Scholastic Inc.,"['Fantasy' 'Young Adult' 'Fiction' 'Fantasy,Magic' 'Childrens' 'Adventure' 'Audiobook' 'Childrens,Middle Grade' 'Classics' 'Science Fiction Fantasy']",89.80,2004,2000,22 anos,"97,871",Mainstream Hit
2,4,Harry Potter and the Chamber of Secrets (Harry Potter #2),J.K. Rowling,4.42,0439554896,eng,352 págs,"6,333",244,2003-11-01 00:00:00,Scholastic,"['Fantasy' 'Fiction' 'Young Adult' 'Fantasy,Magic' 'Childrens' 'Childrens,Middle Grade' 'Audiobook' 'Adventure' 'Classics' 'Science Fiction Fantasy']",88.40,2003,2000,23 anos,275,Mainstream Hit
3,5,Harry Potter and the Prisoner of Azkaban (Harry Potter #3),J.K. Rowling/Mary GrandPré,4.56,043965548X,eng,435 págs,"2,339,585",36325,2004-05-01 00:00:00,Scholastic Inc.,"['Fantasy' 'Fiction' 'Young Adult' 'Fantasy,Magic' 'Childrens' 'Childrens,Middle Grade' 'Adventure' 'Audiobook' 'Classics' 'Science Fiction Fantasy']",91.20,2004,2000,22 anos,"106,345",Mainstream Hit
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potter #1-5),J.K. Rowling/Mary GrandPré,4.78,0439682584,eng,"2,690 págs","41,428",164,2004-09-13 00:00:00,Scholastic,"['Fantasy' 'Young Adult' 'Fiction' 'Fantasy,Magic' 'Adventure' 'Fantasy,Supernatural' 'Mystery' 'Childrens' 'Fantasy,Paranormal' 'Childrens,Middle Grade']",95.60,2004,2000,22 anos,"1,883",Mainstream Hit
5,9,"Unauthorized Harry Potter Book Seven News: ""Half-Blood Prince"" Analysis and Speculation",W. Frederick Zimmerman,3.74,0976540606,en-US,152 págs,19,1,2005-04-26 00:00:00,Nimble Books,['Gênero Não Identificado'],74.80,2005,2000,21 anos,1,Nicho
6,10,Harry Potter Collection (Harry Potter #1-6),J.K. Rowling,4.73,0439827604,eng,"3,342 págs","28,242",808,2005-09-12 00:00:00,Scholastic,"['Fantasy' 'Fiction' 'Young Adult' 'Fantasy,Magic' 'Childrens' 'Classics' 'Adventure' 'Science Fiction Fantasy' 'Novels' 'Paranormal,Wizards']",94.60,2005,2000,21 anos,"1,345",Mainstream Hit
7,12,The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy #1-5),Douglas Adams,4.38,0517226952,eng,815 págs,"3,628",254,2005-11-01 00:00:00,Gramercy Books,"['Science Fiction' 'Fiction' 'Humor' 'Fantasy' 'Classics' 'Humor,Comedy' 'Science Fiction Fantasy' 'Adventure' 'Novels' 'European Literature,British Literature']",87.60,2005,2000,21 anos,173,Mainstream Hit
8,13,The Ultimate Hitchhiker's Guide to the Galaxy (Hitchhiker's Guide to the Galaxy #1-5),Douglas Adams,4.38,0345453743,eng,815 págs,"249,558",4080,2002-04-30 00:00:00,Del Rey Books,"['Science Fiction' 'Fiction' 'Humor' 'Fantasy' 'Classics' 'Humor,Comedy' 'Science Fiction Fantasy' 'Adventure' 'Novels' 'European Literature,British Literature']",87.60,2002,2000,24 anos,"10,398",Mainstream Hit
9,14,The Hitchhiker's Guide to the Galaxy (Hitchhiker's Guide to the Galaxy #1),Douglas Adams,4.22,1400052920,eng,215 págs,"4,930",460,2004-08-03 00:00:00,Crown,"['Science Fiction' 'Fiction' 'Humor' 'Classics' 'Fantasy' 'Humor,Comedy' 'Science Fiction Fantasy' 'Audiobook' 'Adventure' 'Novels']",84.40,2004,2000,22 anos,224,Mainstream Hit


---
## Salvando DataSet

In [6]:
# Salvando DataFrame para uso futuro
ds.save_dataset(
    df=df_books,
    pasta='../../data/interim/books',
    nome_arquivo='goodreads_books_enriched', 
    tipo_arquivo='parquet'
)

Sucesso! Ficheiro guardado em '..\..\data\interim\books\goodreads_books_enriched.parquet'


---
## Atomização da Coluna `genres`: Habilitando Granularidade Analítica

O dataset limpo armazena múltiplos gêneros por livro em uma única célula delimitada por ponto-e-vírgula — um formato conveniente para armazenamento, mas **incompatível com análises por categoria**. Para responder perguntas como "qual gênero tem a maior média de avaliação?" ou "quais gêneros dominam em número de publicações?", é necessário normalizar a relação livro-gênero para um modelo one-to-many explícito.

- **Decisão arquitetural:** O modelo "explodido" (uma linha por par livro-gênero) é a forma canônica para análises de frequência e performance por categoria em dados de consumo cultural
- **Trade-off aceito:** O dataset resultante terá mais linhas que o original; isso é esperado e desejado — cada linha agora representa uma relação analítica atômica

In [7]:
df_exploded = feat.explodir_dataset(df_books, 'genres')

In [8]:
# Validação da granularidade atômica: uma linha por par livro-gênero
display(tb.estilizar_tabela(
    df=df_exploded,
    colunas_selecionadas=['Title', 'genres'],
    qtd_linhas=20,
    caption="DataFrame Explodido"
))

,genres
0,Fantasy
1,Young Adult
2,Fiction
3,"Fantasy,Magic"
4,Childrens
5,Adventure
6,Audiobook
7,"Childrens,Middle Grade"
8,Classics
9,Science Fiction Fantasy


In [9]:
df_exploded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94023 entries, 0 to 94022
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   book_id             94023 non-null  int64         
 1   title               94023 non-null  object        
 2   author              94023 non-null  object        
 3   average_rating      94023 non-null  float64       
 4   isbn                94023 non-null  object        
 5   original_language   94023 non-null  object        
 6   num_pages           94023 non-null  int64         
 7   total_votes         94023 non-null  int64         
 8   text_reviews_count  94023 non-null  int64         
 9   release_date        94023 non-null  datetime64[ns]
 10  producer_company    94023 non-null  object        
 11  genres              94023 non-null  object        
 12  global_score        94023 non-null  float64       
 13  release_year        94023 non-null  int32     

---
## Persistência do Dataset Desnormalizado

O DataFrame explodido é salvo como um artefato independente para preservar o dataset enriquecido sem sobrescrever o arquivo limpo da etapa anterior. Essa separação respeita a **imutabilidade das camadas do pipeline**: `cleaned` permanece como fonte de verdade para transformações alternativas, enquanto `exploded` serve como insumo direto para os notebooks de análise.

In [10]:
# Salvando DataFrame explodido para uso futuro
ds.save_dataset(
    df=df_exploded,
    pasta='../../data/interim/books',
    nome_arquivo='goodreads_books_exploded', 
    tipo_arquivo='parquet'
)

Sucesso! Ficheiro guardado em '..\..\data\interim\books\goodreads_books_exploded.parquet'


---
## Conclusão da Engenharia de Features

Nesta etapa, o dataset previamente limpo foi transformado e modelado para suportar consultas analíticas avançadas e extração de *insights* precisos. As principais refatorações incluíram:

* **Atomização Categórica (Explode):** A coluna `genres` foi fatiada e desmembrada, alterando a granularidade do dataset. Agora, o modelo suporta relações N:M, viabilizando análises estatísticas isoladas para cada gênero literário sem o mascaramento de dados agrupados.
* **Inteligência Temporal:** Estruturação de dados de data (`datetime`) para permitir futuras análises de tendências de publicação e engajamento ao longo do tempo.

**Exportação Estratégica:**
O dataset refatorado foi exportado com sucesso para:
  -  `data/processed/goodreads_genres_exploded.csv`  
  -  `data/processed/goodreads_genres_exploded.pkl`


> **Nota Arquitetural:** Para a próxima fase, utilizaremos uma abordagem de *Dual-Dataset*. O arquivo recém-exportado será dedicado exclusivamente às análises de gênero, enquanto o arquivo `goodreads_cleaned.csv` será mantido para métricas de volume geral (evitando a contagem duplicada de obras).

**Próximo Passo:**
Com as bases prontas e a arquitetura definida, o projeto avança para a **Análise Exploratória de Dados (EDA)** para investigar os direcionadores de notas e popularidade das obras.

---